In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import sys
from pathlib import Path

In [3]:
PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(PROJECT_ROOT)

/home/abril/UdeSA/ML/TP_Final_ML


In [4]:
from category_utils import (
    load_dataset,
    get_categorical_columns,
    build_category_summary,
    build_category_table,
    save_category_audit_report,
)

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Categories Per Feature Analysis
</h3>

Este notebook tiene como objetivo auditar las variables categóricas del dataset de SUVs. La idea es identificar qué categorías existen en cada columna, cuántas veces aparece cada una y qué porcentaje representan dentro de la variable.

Este análisis ayuda a entender con qué datos estamos trabajando antes de limpiar, agrupar categorías raras o decidir qué variables usar en el modelo.

El notebook genera un archivo HTML con el reporte completo de categorías. Esto es útil porque Jupyter suele recortar tablas muy grandes, mientras que el HTML permite revisar todas las categorías con más comodidad y buscar valores usando `Ctrl + F`.

El archivo se guarda en la carpeta:

`html/category_audit_report.html`

Para abrirlo, ir a esa carpeta desde el explorador de archivos o desde VS Code, hacer click derecho sobre `category_audit_report.html` y elegir abrirlo en el navegador. También se puede abrir manualmente copiando la ruta completa del archivo en Chrome o Firefox.

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Load Dataset
</div>

In [5]:
df = load_dataset()

display(df.head())
display(df.shape)

,Marca,Modelo,Año,Versión,Color,Tipo de combustible,Puertas,Transmisión,Motor,Tipo de carrocería,Con cámara de retroceso,Kilómetros,Título,Precio,Moneda,Descripción,Tipo de vendedor
0,Ford,Ecosport,2020.0,1.5 Freestyle 123cv 4x2,Blanco,Nafta,5.0,Manual,1.5,SUV,No,64000.0,Ford Ecosport 1.5 Freestyle 123cv 4x2,20500000.0,$,Descubre nuestro impresionante Ford Eco Sport ...,concesionaria
1,Volkswagen,Tiguan,2024.0,LIFE 350 TSI 4M,Negro,Nafta,5.0,Automática secuencial,2.0 L 230 CV 350 TSI,SUV,Sí,0.0,Tiguan Life 350 Tsi 4m Ar,55999900.0,$,AUTOTAG S.A. Concesionario Oficial N°1 Volkswa...,tienda
2,Volkswagen,Tiguan Allspace,2019.0,1.4 Tsi Trendline 150cv Dsg,Negro,Nafta,5.0,Automática,1.4,SUV,NaN,65300.0,Volkswagen Tiguan Allspace 1.4 Tsi Trendline 1...,28300.0,US$,-HERMOSA TIGUAN 1.4 TSI DSG-MANTENIMIENTO/SERV...,particular
3,Ford,Ecosport,2017.0,1.5 Titanium 123Cv 4X2,Blanco,Nafta,5.0,Manual,1.5,SUV,NaN,76898.0,Ford Ecosport 1.5 Titanium 123Cv 4X2,20090000.0,$,"En GRUPO RANDAZZO, tenemos el auto que estas b...",tienda
4,Jeep,Compass,2021.0,2.4 Sport At,NaN,Nafta,5.0,Automática,2.4,SUV,NaN,109000.0,Jeep Compass 2.4 Sport At,23000.0,US$,•Unico dueño•Todos los services oficiales•Bate...,particular


(18254, 17)

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Categorical Columns
</div>

In [6]:
categorical_columns = get_categorical_columns(df)

category_summary = (
    df[categorical_columns]
    .nunique(dropna=False)
    .sort_values(ascending=False)
    .reset_index()
)

category_summary.columns = ["feature", "unique_categories"]

display(category_summary)

,feature,unique_categories
0,Descripción,12142
1,Título,2220
2,Kilómetros,2175
3,Versión,2072
4,Motor,272
5,Modelo,137
6,Color,71
7,Marca,47
8,Tipo de combustible,8
9,Transmisión,5


<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Category Summary
</div>

In [7]:
ignored_columns = ["Título", "Descripción"]

categorical_columns = get_categorical_columns(df, ignored_columns=ignored_columns,)
categorical_columns

['Marca',
 'Modelo',
 'Versión',
 'Color',
 'Tipo de combustible',
 'Transmisión',
 'Motor',
 'Tipo de carrocería',
 'Con cámara de retroceso',
 'Kilómetros',
 'Moneda',
 'Tipo de vendedor']

In [8]:
category_summary = build_category_summary(df, categorical_columns)

display(category_summary)

,feature,unique_categories
0,Kilómetros,2175
1,Versión,2072
2,Motor,272
3,Modelo,137
4,Color,71
5,Marca,47
6,Tipo de combustible,8
7,Transmisión,5
8,Tipo de vendedor,3
9,Con cámara de retroceso,3


<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Full Category Table Preview
</div>

In [9]:
category_table = build_category_table(df, categorical_columns)

display(category_table.head(50))

,feature,rank,category,count,percentage
0,Marca,1,Ford,2161,11.84
1,Marca,2,Jeep,2050,11.23
2,Marca,3,Volkswagen,2037,11.16
3,Marca,4,Chevrolet,1750,9.59
4,Marca,5,Renault,1491,8.17
5,Marca,6,Toyota,1260,6.90
6,Marca,7,Peugeot,1250,6.85
7,Marca,8,Nissan,1059,5.80
8,Marca,9,Citroën,721,3.95
9,Marca,10,BMW,672,3.68


<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Rare Categories
</div>

In [10]:
display(category_table[category_table["count"] <= 5] .sort_values(["feature", "count", "category"]))

,feature,rank,category,count,percentage
2307,Color,52,Amarrillo,1,0.01
2302,Color,47,BLACK MEET KETTLE,1,0.01
2325,Color,70,BLANCO BANCHISA BICOLOR NEGRO,1,0.01
2313,Color,58,BLANCO BANQUISE,1,0.01
2319,Color,64,BLANCO GLACIAR,1,0.01
...,...,...,...,...,...
700,Versión,517,4.2 V8 Premium,5,0.03
709,Versión,526,45 TFSI ADVANCED STRONIC QUATTRO,5,0.03
724,Versión,541,Premier,5,0.03
702,Versión,519,Serie S,5,0.03


<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Save HTML Report
</div>

In [11]:
report_path = save_category_audit_report(category_summary, category_table, output_path="html/category_audit_report.html",)

print(f"Report saved to: {report_path}")

Report saved to: /home/abril/UdeSA/ML/TP_Final_ML/html/category_audit_report.html
